# 02 — Detector API (FastAPI) on Colab

Serves the **third-party** Dhwani ONNX checkpoint
([`ayush2635/Dhwani-Multilingual-Deepfake-Audio-Detection-Model`](https://huggingface.co/ayush2635/Dhwani-Multilingual-Deepfake-Audio-Detection-Model),
MIT, revision `ee01871`, built on `facebook/wav2vec2-xls-r-300m`). We did not train it.

- `GET /health`: model readiness (no key needed, no audio).
- `POST /detect`: multipart field `file`, header `X-API-Key`, optional `?mode=train_peak|train_zscore|api_replica`.
- The response has a prediction, an **uncalibrated** synthetic score (not a probability), duration, timings, the preprocessing mode and clear rejection reasons.
- `train_peak` is the *provisional* default. The two-clip smoke test (run 20261024T102407Z) does not validate it.
- Model loaded once at server start. One inference at a time (lock + single worker). Uploads capped at 8 MiB and deleted after each request.

**Runtime note.** Colab gives every notebook its own runtime (VM), so notebook 01's Hugging Face cache is reused only when
these cells run in that same runtime. In a fresh runtime, step 4 downloads the pinned checkpoint **once** (1.26 GB, ~1 min),
or copies it from Drive if `USE_DRIVE_CACHE = True` and it is there. It is never downloaded per request.

**Order:** 1 → 8 start and test the server locally inside Colab. Step 9 opens a public HTTPS tunnel for the phones. Step 11 stops everything.

In [ ]:
# 1. Dependencies
!pip -q install fastapi uvicorn python-multipart onnxruntime librosa soundfile huggingface_hub requests
import fastapi, uvicorn, onnxruntime
print("fastapi", fastapi.__version__, "| uvicorn", uvicorn.__version__, "| onnxruntime", onnxruntime.__version__)

In [ ]:
# 2. Settings
import os
APP_DIR = "/content/vcd_app"            # API source files
TMP_DIR = "/content/vcd_api_tmp"        # uploads live here only during a request
TEST_DIR = "/content/vcd_api_test"      # test clips
PORT = 8000
DEFAULT_MODE = "train_peak"             # provisional; see notebook 01 comparison
MAX_UPLOAD_MB = 8
USE_DRIVE_CACHE = False                 # True: reuse/keep the checkpoint in Drive
DRIVE_CACHE_DIR = "/content/drive/MyDrive/vcd_cache"
for d in (APP_DIR, TMP_DIR, TEST_DIR):
    os.makedirs(d, exist_ok=True)

In [ ]:
%%writefile /content/vcd_app/dhwani_infer.py
"""Inference wrapper for the third-party Dhwani ONNX checkpoint.

Model: ayush2635/Dhwani-Multilingual-Deepfake-Audio-Detection-Model (MIT),
file best_model.onnx at HF revision MODEL_REVISION. We did NOT train it.

Preprocessing is reproduced from the upstream training notebook
(github.com/ayush2635/Dhwani-Deepfake-Audio-Detection-API @ efd512c,
training/training_notebook.ipynb cell 4), which is the notebook whose cell 14
exports best_model.onnx with a (1, 64000) dummy input. See
docs/UPSTREAM_AUDIT.md for why we do not use the model card's z-score recipe
as the default.

Scores are softmax outputs of an UNCALIBRATED classifier. They are model
scores, not probabilities that a clip is synthetic.
"""

from __future__ import annotations

import hashlib
import json
import time
from dataclasses import dataclass, field, asdict

import numpy as np

# ---- pinned provenance -----------------------------------------------------
MODEL_REPO = "ayush2635/Dhwani-Multilingual-Deepfake-Audio-Detection-Model"
MODEL_FILE = "best_model.onnx"
MODEL_REVISION = "ee0187142973e1f02050e47d2a31c55ee0263065"
MODEL_SHA256 = "d1c232bf4d7990526804b375c15804c0ee4e9b566478b908c5a57431491b7342"
MODEL_BYTES = 1262760439

# ---- preprocessing / windowing policy ---------------------------------------
SR = 16000
WIN = 64000            # 4.0 s: training crop length and ONNX export shape
HOP = 32000            # 2.0 s: same hop as the upstream API's sliding window
MIN_DUR_S = 2.0        # brief: 2-10 s clips
MAX_DUR_S = 10.0       # longer clips: analyse the first 10 s, flagged
PEAK_TARGET_DBFS = -3.0

# ---- rejection gates (input validation only; never fed to the model) --------
SILENT_PEAK_DBFS = -50.0   # whole clip quieter than this -> silent
ACTIVE_FRAME_S = 0.025
ACTIVE_REL_DB = 35.0       # frame counts as active if within 35 dB of loudest frame
ACTIVE_ABS_DBFS = -55.0    # ...and above this absolute level
MIN_ACTIVE_S = 1.0

# Class order from upstream dataset.py: labels = [0]*real + [1]*fake.
# Documented in source; must still be confirmed empirically on known clips.
LABELS = {0: "genuine", 1: "synthetic"}
SYNTHETIC_INDEX = 1
DECISION_THRESHOLD = 0.5   # placeholder, NOT tuned on any validation set

# Controlled comparison (docs/UPSTREAM_AUDIT.md):
#   train_peak   - training recipe: peak -3 dBFS, 4 s windows             (A)
#   train_zscore - identical windows, z-score instead of peak norm        (A vs B isolates normalisation)
#   api_replica  - deployed API: first 3 s, z-score, zero-pad to 4 s     (B vs C isolates crop/padding)
PREPROCESSING_MODES = ("train_peak", "train_zscore", "api_replica")


class AudioRejected(Exception):
    """Input cannot be analysed (unreadable, empty, silent, too short)."""


def sha256_file(path: str, chunk: int = 1 << 22) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def _dbfs(x: float) -> float:
    return float(20 * np.log10(max(x, 1e-12)))


# ---- loading ------------------------------------------------------------------
def load_audio(path: str):
    """Return (samples [channels, n] float32 at native rate, native_sr).

    Tries soundfile first (wav/flac/ogg), then librosa/audioread (mp3, m4a, opus
    via ffmpeg). Never substitutes zeros on failure.
    """
    import soundfile as sf

    try:
        y, sr = sf.read(path, dtype="float32", always_2d=True)  # (n, ch)
        y = y.T
    except Exception as sf_err:
        try:
            import warnings

            import librosa

            with warnings.catch_warnings():  # expected audioread fallback for m4a/aac
                warnings.simplefilter("ignore", (UserWarning, FutureWarning))
                y, sr = librosa.load(path, sr=None, mono=False)
            y = np.atleast_2d(y).astype(np.float32)
        except Exception as lr_err:
            raise AudioRejected(
                f"unreadable audio (soundfile: {type(sf_err).__name__}; "
                f"librosa: {type(lr_err).__name__}: {lr_err})"
            ) from lr_err
    if y.size == 0 or y.shape[-1] == 0:
        raise AudioRejected("empty audio (0 samples)")
    if not np.all(np.isfinite(y)):
        raise AudioRejected("audio contains NaN/Inf samples")
    return y, int(sr)


def to_mono_16k(y: np.ndarray, sr: int) -> np.ndarray:
    mono = y.mean(axis=0) if y.shape[0] > 1 else y[0]
    if sr != SR:
        import librosa

        mono = librosa.resample(mono, orig_sr=sr, target_sr=SR, res_type="soxr_hq")
    return mono.astype(np.float32)


def signal_stats(x: np.ndarray) -> dict:
    peak = float(np.max(np.abs(x))) if x.size else 0.0
    rms = float(np.sqrt(np.mean(x.astype(np.float64) ** 2))) if x.size else 0.0
    flen = int(ACTIVE_FRAME_S * SR)
    nfr = len(x) // flen
    if nfr:
        fr = x[: nfr * flen].reshape(nfr, flen).astype(np.float64)
        fdb = 10 * np.log10(np.mean(fr**2, axis=1) + 1e-20)
        active = (fdb > fdb.max() - ACTIVE_REL_DB) & (fdb > ACTIVE_ABS_DBFS)
        active_s = float(active.sum() * ACTIVE_FRAME_S)
    else:
        active_s = 0.0
    return {
        "peak_dbfs": round(_dbfs(peak), 2),
        "rms_dbfs": round(_dbfs(rms), 2),
        "active_s": round(active_s, 3),
        "clipping_frac": round(float(np.mean(np.abs(x) >= 0.999)) if x.size else 0.0, 5),
    }


def validate(x16: np.ndarray) -> tuple[np.ndarray, dict]:
    """Apply the 2-10 s policy and silence gates. Returns (analysed audio, info)."""
    dur = len(x16) / SR
    info = {"duration_s": round(dur, 3), "truncated": False}
    if dur < MIN_DUR_S:
        raise AudioRejected(f"too short: {dur:.2f} s < {MIN_DUR_S} s minimum")
    if dur > MAX_DUR_S:
        x16 = x16[: int(MAX_DUR_S * SR)]
        info["truncated"] = True
    info["analysed_duration_s"] = round(len(x16) / SR, 3)
    st = signal_stats(x16)
    info.update(st)
    if st["peak_dbfs"] < SILENT_PEAK_DBFS:
        raise AudioRejected(f"silent: peak {st['peak_dbfs']} dBFS < {SILENT_PEAK_DBFS} dBFS")
    if st["active_s"] < MIN_ACTIVE_S:
        raise AudioRejected(
            f"mostly silent: {st['active_s']:.2f} s active signal < {MIN_ACTIVE_S} s"
        )
    return x16, info


# ---- preprocessing --------------------------------------------------------------
def window_starts(n: int) -> list[int]:
    """4 s windows, 2 s hop, last window aligned to the clip end (full coverage)."""
    if n <= WIN:
        return [0]
    starts = list(range(0, n - WIN + 1, HOP))
    if starts[-1] + WIN < n:
        starts.append(n - WIN)
    return starts


def make_windows(x16: np.ndarray, mode: str) -> tuple[np.ndarray, list[int]]:
    """Return (windows [k, WIN] float32, start sample indices)."""
    if mode in ("train_peak", "train_zscore"):
        if mode == "train_peak":
            # Training notebook: peak-normalise the clip to -3 dBFS, then crop/zero-pad
            # the END to 64000 samples. No mean/variance normalisation.
            peak = float(np.max(np.abs(x16)))
            x = x16 * ((10 ** (PEAK_TARGET_DBFS / 20)) / (peak + 1e-6))
        else:
            # Same windows; z-score over the analysed clip (before padding).
            x = (x16 - x16.mean()) / np.sqrt(x16.var() + 1e-5)
        starts = window_starts(len(x))
        wins = []
        for s in starts:
            w = x[s : s + WIN]
            if len(w) < WIN:
                w = np.pad(w, (0, WIN - len(w)))
            wins.append(w)
        return np.stack(wins).astype(np.float32), starts
    if mode == "api_replica":
        # Deployed upstream API (src/api/inference.py): first 48000 samples,
        # zero-pad to 48000, z-score, then zero-pad to 64000 in predict().
        # Kept only as a comparison; it differs from training.
        w = x16[:48000]
        w = np.pad(w, (0, 48000 - len(w)))
        w = (w - w.mean()) / np.sqrt(w.var() + 1e-5)
        w = np.pad(w, (0, WIN - len(w)))
        return w[None, :].astype(np.float32), [0]
    raise ValueError(f"unknown preprocessing mode {mode!r}; use {PREPROCESSING_MODES}")


def softmax(z: np.ndarray) -> np.ndarray:
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)


# ---- model ------------------------------------------------------------------------
def load_session(model_path: str, verify_sha: bool = True):
    import onnxruntime as ort

    if verify_sha:
        got = sha256_file(model_path)
        if got != MODEL_SHA256:
            raise RuntimeError(f"model sha256 mismatch: {got} != pinned {MODEL_SHA256}")
    so = ort.SessionOptions()
    sess = ort.InferenceSession(model_path, sess_options=so, providers=["CPUExecutionProvider"])
    return sess


def describe_session(sess) -> dict:
    import onnxruntime as ort

    return {
        "onnxruntime_version": ort.__version__,
        "available_providers": ort.get_available_providers(),
        "session_providers": sess.get_providers(),
        "inputs": [(i.name, i.shape, i.type) for i in sess.get_inputs()],
        "outputs": [(o.name, o.shape, o.type) for o in sess.get_outputs()],
    }


def interface_self_test(sess, seed: int = 0) -> dict:
    """Probe what the checkpoint actually accepts (metadata alone proves nothing)."""
    rng = np.random.default_rng(seed)
    name = sess.get_inputs()[0].name
    out = {}
    base = (rng.standard_normal(WIN) * 0.1).astype(np.float32)
    ref = sess.run(None, {name: base[None]})[0]
    out["len_64000_batch1"] = {"ok": True, "shape": list(ref.shape)}
    for n in (32000, 48000, 96000):
        try:
            r = sess.run(None, {name: base[:n][None] if n <= WIN else np.tile(base, 2)[:n][None]})[0]
            out[f"len_{n}_batch1"] = {"ok": True, "shape": list(r.shape)}
        except Exception as e:
            out[f"len_{n}_batch1"] = {"ok": False, "error": f"{type(e).__name__}: {str(e)[:200]}"}
    try:
        b = np.stack([base, base[::-1].copy()])
        r = sess.run(None, {name: b})[0]
        r1 = sess.run(None, {name: base[::-1].copy()[None]})[0]
        # Colab run 20261024T102407Z: batch of 2 returned shape [1, 2] (a row is
        # silently dropped), so "ok" requires one output row per input row.
        out["batch2"] = {
            "ok": r.shape[0] == 2,
            "shape": list(r.shape),
            "max_abs_diff_vs_individual": float(
                max(np.abs(r[0] - ref[0]).max(), np.abs(r[1] - r1[0]).max())
            ) if r.shape[0] == 2 else None,
        }
    except Exception as e:
        out["batch2"] = {"ok": False, "error": f"{type(e).__name__}: {str(e)[:200]}"}
    # determinism
    again = sess.run(None, {name: base[None]})[0]
    out["deterministic"] = bool(np.allclose(again, ref))
    return out


@dataclass
class Result:
    file: str
    status: str = "ok"
    reject_reason: str = ""
    file_sha256: str = ""
    native_sr: int | None = None
    channels: int | None = None
    info: dict = field(default_factory=dict)
    preprocessing: str = ""
    window_starts_s: list = field(default_factory=list)
    window_logits: list = field(default_factory=list)
    window_scores: list = field(default_factory=list)
    clip_score_mean: float | None = None
    clip_score_max: float | None = None
    predicted_label: str = ""
    latency_ms_model: float | None = None
    latency_ms_total: float | None = None


def analyse(sess, path: str, mode: str = "train_peak") -> Result:
    """Score an audio file (any format load_audio can decode)."""
    t0 = time.perf_counter()
    res = Result(file=path, preprocessing=mode)
    try:
        res.file_sha256 = sha256_file(path)
        y, sr = load_audio(path)
    except AudioRejected as e:
        res.status, res.reject_reason = "rejected", str(e)
        res.latency_ms_total = round((time.perf_counter() - t0) * 1000, 1)
        return res
    return _score(sess, res, y, sr, t0)


def analyse_waveform(sess, samples: np.ndarray, sr: int, mode: str = "train_peak",
                     label: str = "<waveform>") -> Result:
    """Score in-memory PCM ([n] or [channels, n], float in [-1, 1]).

    Same validation, preprocessing and windowing as analyse(); intended for
    audio windows that never touch disk (e.g. remote-call PCM later).
    """
    t0 = time.perf_counter()
    res = Result(file=label, preprocessing=mode)
    y = np.atleast_2d(np.asarray(samples, dtype=np.float32))
    if y.size == 0:
        res.status, res.reject_reason = "rejected", "empty audio (0 samples)"
    elif not np.all(np.isfinite(y)):
        res.status, res.reject_reason = "rejected", "audio contains NaN/Inf samples"
    if res.status == "rejected":
        res.latency_ms_total = round((time.perf_counter() - t0) * 1000, 1)
        return res
    return _score(sess, res, y, int(sr), t0)


def _score(sess, res: Result, y: np.ndarray, sr: int, t0: float) -> Result:
    try:
        res.native_sr, res.channels = sr, int(y.shape[0])
        x16 = to_mono_16k(y, sr)
        x16, info = validate(x16)
        res.info = info
        wins, starts = make_windows(x16, res.preprocessing)
    except AudioRejected as e:
        res.status, res.reject_reason = "rejected", str(e)
        res.latency_ms_total = round((time.perf_counter() - t0) * 1000, 1)
        return res

    name = sess.get_inputs()[0].name
    logits = []
    tm = time.perf_counter()
    for w in wins:  # batch=1 only: batch>1 drops rows (see interface_self_test)
        logits.append(sess.run(None, {name: w[None]})[0][0])
    res.latency_ms_model = round((time.perf_counter() - tm) * 1000, 1)
    logits = np.asarray(logits, dtype=np.float64)
    scores = softmax(logits)[:, SYNTHETIC_INDEX]
    res.window_starts_s = [round(s / SR, 3) for s in starts]
    res.window_logits = [[round(float(a), 4) for a in row] for row in logits]
    res.window_scores = [round(float(s), 4) for s in scores]
    res.clip_score_mean = round(float(scores.mean()), 4)
    res.clip_score_max = round(float(scores.max()), 4)
    res.predicted_label = LABELS[SYNTHETIC_INDEX] if res.clip_score_mean >= DECISION_THRESHOLD else LABELS[0]
    res.latency_ms_total = round((time.perf_counter() - t0) * 1000, 1)
    return res


def result_row(res: Result, **extra) -> dict:
    d = asdict(res)
    info = d.pop("info")
    for k in ("window_starts_s", "window_logits", "window_scores"):
        d[k] = json.dumps(d[k])
    row = {**extra, **d, **info}
    row.update(
        decision_threshold=DECISION_THRESHOLD,
        label_mapping=json.dumps(LABELS),
        model_repo=MODEL_REPO,
        model_revision=MODEL_REVISION,
        model_sha256=MODEL_SHA256,
    )
    return row

In [ ]:
%%writefile /content/vcd_app/vcd_service.py
"""Detector service: model lifecycle + serialised inference. No HTTP here.

Wraps the third-party Dhwani ONNX checkpoint via dhwani_infer (pinned repo,
revision and sha256 live there). Used by vcd_api (FastAPI) now and intended
for remote-call audio windows later (detect_waveform).
"""

from __future__ import annotations

import os
import threading
import time

import numpy as np

import dhwani_infer as dh

ATTRIBUTION = {
    "model": dh.MODEL_REPO,
    "revision": dh.MODEL_REVISION,
    "sha256": dh.MODEL_SHA256,
    "licence": "MIT (per model card)",
    "authors": "ayush2635 (Dhwani, HCL Guvi Hackathon); not trained by this project",
    "upstream_code": "github.com/ayush2635/Dhwani-Deepfake-Audio-Detection-API@efd512c",
    "encoder": "facebook/wav2vec2-xls-r-300m (Apache-2.0)",
}
SCORE_NOTE = (
    "synthetic_score is the mean softmax output for the synthetic class over 4 s windows. "
    "It is uncalibrated: a model score, not a probability. The 0.5 threshold is a placeholder."
)
MODE_NOTE = (
    "train_peak is a provisional default (matches upstream training code); "
    "it has not been validated beyond a two-clip smoke test."
)


class ServiceBusy(Exception):
    """Another request held the inference slot longer than the wait limit."""


class ServiceNotReady(Exception):
    """Model failed to load or has not loaded yet."""


def resolve_model_path(allow_download: bool = False) -> str:
    """Pinned checkpoint path: VCD_MODEL_PATH, else the local HF cache.

    Downloads only if allow_download is set (once, at startup - never per request).
    """
    explicit = os.environ.get("VCD_MODEL_PATH")
    if explicit:
        return explicit
    from huggingface_hub import hf_hub_download

    try:
        return hf_hub_download(dh.MODEL_REPO, dh.MODEL_FILE, revision=dh.MODEL_REVISION,
                               local_files_only=True)
    except Exception:
        if not allow_download:
            raise FileNotFoundError(
                "pinned checkpoint not in local HF cache; set VCD_MODEL_PATH or VCD_ALLOW_DOWNLOAD=1"
            )
    return hf_hub_download(dh.MODEL_REPO, dh.MODEL_FILE, revision=dh.MODEL_REVISION)


def verify_checkpoint(path: str) -> str:
    """Size + sha256 check; reuses the notebook's verification marker when the file is unchanged."""
    size = os.path.getsize(path)
    if size != dh.MODEL_BYTES:
        raise RuntimeError(f"unexpected checkpoint size {size} != {dh.MODEL_BYTES}")
    marker = os.path.join(os.path.dirname(os.path.realpath(path)), ".dhwani_sha256_ok")
    stamp = f"{size}:{os.path.getmtime(path)}"
    if os.path.exists(marker) and open(marker).read() == stamp:
        return "previously verified"
    got = dh.sha256_file(path)
    if got != dh.MODEL_SHA256:
        raise RuntimeError(f"checkpoint sha256 mismatch: {got}")
    try:
        with open(marker, "w") as f:
            f.write(stamp)
    except OSError:
        pass
    return "verified now"


class DetectorService:
    def __init__(self, default_mode: str = "train_peak", allow_download: bool = False,
                 max_wait_s: float = 60.0):
        if default_mode not in dh.PREPROCESSING_MODES:
            raise ValueError(f"unknown preprocessing mode {default_mode!r}")
        self.default_mode = default_mode
        self.allow_download = allow_download
        self.max_wait_s = max_wait_s
        self._lock = threading.Lock()  # one inference at a time: avoids overlapping CPU work
        self.sess = None
        self.model_path = None
        self.load_error = None
        self.load_info = {}

    # ---- lifecycle --------------------------------------------------------------
    def load(self) -> None:
        """Load once. Errors are kept for /health instead of crashing the server."""
        t0 = time.perf_counter()
        try:
            self.model_path = resolve_model_path(self.allow_download)
            sha_status = verify_checkpoint(self.model_path)
            self.sess = dh.load_session(self.model_path, verify_sha=False)
            warm = (np.random.default_rng(0).standard_normal(dh.WIN) * 0.1).astype(np.float32)
            tw = time.perf_counter()
            self.sess.run(None, {self.sess.get_inputs()[0].name: warm[None]})
            self.load_info = {
                "sha256_check": sha_status,
                "load_s": round(time.perf_counter() - t0, 2),
                "warmup_window_ms": round((time.perf_counter() - tw) * 1000, 1),
                **dh.describe_session(self.sess),
            }
        except Exception as e:
            self.sess = None
            self.load_error = f"{type(e).__name__}: {e}"

    @property
    def ready(self) -> bool:
        return self.sess is not None

    def health(self) -> dict:
        return {
            "status": "ready" if self.ready else "not_ready",
            "model_ready": self.ready,
            "load_error": self.load_error,
            "default_preprocessing": self.default_mode,
            "preprocessing_modes": list(dh.PREPROCESSING_MODES),
            "busy": self._lock.locked(),
            "model": ATTRIBUTION,
            "load_info": {k: self.load_info[k] for k in
                          ("sha256_check", "load_s", "warmup_window_ms", "session_providers")
                          if k in self.load_info},
        }

    # ---- inference ----------------------------------------------------------------
    def detect_file(self, path: str, mode: str | None = None) -> dict:
        mode = self._mode(mode)
        return self._serialised(lambda: dh.analyse(self.sess, path, mode=mode))

    def detect_waveform(self, samples: np.ndarray, sr: int, mode: str | None = None) -> dict:
        mode = self._mode(mode)
        return self._serialised(lambda: dh.analyse_waveform(self.sess, samples, sr, mode=mode))

    def _mode(self, mode: str | None) -> str:
        mode = mode or self.default_mode
        if mode not in dh.PREPROCESSING_MODES:
            raise ValueError(f"unknown preprocessing mode {mode!r}; use one of {list(dh.PREPROCESSING_MODES)}")
        return mode

    def _serialised(self, fn) -> dict:
        if not self.ready:
            raise ServiceNotReady(self.load_error or "model not loaded")
        tq = time.perf_counter()
        if not self._lock.acquire(timeout=self.max_wait_s):
            raise ServiceBusy(f"inference slot busy for more than {self.max_wait_s:.0f} s")
        queue_ms = round((time.perf_counter() - tq) * 1000, 1)
        acquired = time.monotonic()
        try:
            res = fn()
        finally:
            released = time.monotonic()
            self._lock.release()
        out = to_response(res)
        out["timing_ms"]["queue_wait"] = queue_ms
        out["inference_slot"] = {"acquired_monotonic": round(acquired, 4), "released_monotonic": round(released, 4)}
        return out


def to_response(res: dh.Result) -> dict:
    ok = res.status == "ok"
    info = res.info or {}
    return {
        "status": res.status,
        "prediction": res.predicted_label if ok else None,
        "synthetic_score": res.clip_score_mean,
        "synthetic_score_max": res.clip_score_max,
        "score_is_calibrated": False,
        "decision_threshold": dh.DECISION_THRESHOLD,
        "threshold_is_tuned": False,
        "score_note": SCORE_NOTE,
        "reject_reason": res.reject_reason or None,
        "duration_s": info.get("duration_s"),
        "analysed_duration_s": info.get("analysed_duration_s"),
        "truncated": info.get("truncated"),
        "audio": {
            "native_sample_rate": res.native_sr,
            "channels": res.channels,
            "peak_dbfs": info.get("peak_dbfs"),
            "rms_dbfs": info.get("rms_dbfs"),
            "active_s": info.get("active_s"),
        },
        "windows": [{"start_s": s, "synthetic_score": w}
                    for s, w in zip(res.window_starts_s, res.window_scores)],
        "preprocessing": res.preprocessing,
        "preprocessing_note": MODE_NOTE,
        "timing_ms": {"model": res.latency_ms_model, "pipeline": res.latency_ms_total},
        "model": {"repo": dh.MODEL_REPO, "revision": dh.MODEL_REVISION},
    }

In [ ]:
%%writefile /content/vcd_app/vcd_api.py
"""HTTP layer (FastAPI) for the voice-clone detector.

All inference logic lives in vcd_service / dhwani_infer; this file only handles
auth, upload limits, temp files and status codes. Model: third-party Dhwani
ONNX checkpoint (MIT), see vcd_service.ATTRIBUTION.

Environment:
  VCD_API_KEY            required, >= 16 chars; sent by clients as X-API-Key
  VCD_MODEL_PATH         pinned checkpoint path (else local HF cache)
  VCD_ALLOW_DOWNLOAD     "1" to allow one download at startup (default off)
  VCD_PREPROCESSING      default mode (train_peak | train_zscore | api_replica)
  VCD_MAX_UPLOAD_BYTES   default 8 MiB
  VCD_TMP_DIR            where uploads are written briefly (deleted after each request)
  VCD_MAX_WAIT_S         max seconds a request waits for the inference slot (default 60)

Run (single worker - one model, one inference at a time):
  uvicorn vcd_api:app --app-dir <dir with these files> --host 127.0.0.1 --port 8000 --workers 1
"""

from __future__ import annotations

import hmac
import logging
import os
import tempfile
import time
import uuid
from contextlib import asynccontextmanager

from fastapi import Depends, FastAPI, File, Header, HTTPException, Query, UploadFile
from fastapi.responses import JSONResponse

from vcd_service import DetectorService, ServiceBusy, ServiceNotReady

API_KEY = os.environ.get("VCD_API_KEY", "")
MAX_UPLOAD_BYTES = int(os.environ.get("VCD_MAX_UPLOAD_BYTES", str(8 * 1024 * 1024)))
MULTIPART_OVERHEAD = 64 * 1024
TMP_DIR = os.environ.get("VCD_TMP_DIR") or os.path.join(tempfile.gettempdir(), "vcd_api_tmp")
ALLOWED_EXT = {".wav", ".m4a", ".mp3", ".aac", ".ogg", ".opus", ".flac", ".3gp", ".amr", ".webm", ".mp4"}

log = logging.getLogger("vcd_api")
service = DetectorService(
    default_mode=os.environ.get("VCD_PREPROCESSING", "train_peak"),
    allow_download=os.environ.get("VCD_ALLOW_DOWNLOAD") == "1",
    max_wait_s=float(os.environ.get("VCD_MAX_WAIT_S", "60")),
)


@asynccontextmanager
async def lifespan(app: FastAPI):
    if len(API_KEY) < 16:
        raise RuntimeError("set VCD_API_KEY (at least 16 characters) before starting the API")
    os.makedirs(TMP_DIR, exist_ok=True)
    service.load()  # once per process; a failure is reported by /health
    log.info("model ready=%s error=%s", service.ready, service.load_error)
    yield


app = FastAPI(title="Voice-clone detector (Dhwani baseline)", version="0.1.0", lifespan=lifespan)


class BodySizeLimit:
    """Reject oversized request bodies before they are spooled to disk."""

    def __init__(self, app, max_bytes: int):
        self.app, self.max_bytes = app, max_bytes

    async def __call__(self, scope, receive, send):
        if scope["type"] != "http":
            return await self.app(scope, receive, send)
        cl = dict(scope.get("headers") or []).get(b"content-length")
        if cl is not None and cl.isdigit() and int(cl) > self.max_bytes:
            resp = JSONResponse(status_code=413, content=_err("upload_too_large",
                                f"request body exceeds {self.max_bytes} bytes"))
            return await resp(scope, receive, send)
        seen = 0

        async def limited_receive():
            nonlocal seen
            msg = await receive()
            if msg["type"] == "http.request":
                seen += len(msg.get("body", b""))
                if seen > self.max_bytes:  # chunked uploads without Content-Length
                    raise HTTPException(413, _err("upload_too_large", f"request body exceeds {self.max_bytes} bytes"))
            return msg

        await self.app(scope, limited_receive, send)


app.add_middleware(BodySizeLimit, max_bytes=MAX_UPLOAD_BYTES + MULTIPART_OVERHEAD)


def _err(code: str, message: str) -> dict:
    return {"status": "error", "error": code, "message": message}


def require_api_key(x_api_key: str | None = Header(default=None)) -> None:
    if not x_api_key or not hmac.compare_digest(x_api_key.encode(), API_KEY.encode()):
        raise HTTPException(401, _err("unauthorized", "missing or invalid X-API-Key header"))


@app.exception_handler(HTTPException)
async def _http_error(request, exc: HTTPException):
    body = exc.detail if isinstance(exc.detail, dict) else _err("http_error", str(exc.detail))
    return JSONResponse(status_code=exc.status_code, content=body, headers=getattr(exc, "headers", None))


@app.get("/health")
def health():
    return service.health()


@app.post("/detect", dependencies=[Depends(require_api_key)])
def detect(file: UploadFile = File(...), mode: str | None = Query(default=None)):
    t0 = time.perf_counter()
    request_id = uuid.uuid4().hex[:12]
    ext = os.path.splitext(file.filename or "")[1].lower()
    if ext not in ALLOWED_EXT:
        raise HTTPException(415, _err("unsupported_type", f"extension {ext or '(none)'} not in {sorted(ALLOWED_EXT)}"))
    if not service.ready:
        raise HTTPException(503, _err("model_not_ready", service.load_error or "model not loaded"))

    fd, tmp_path = tempfile.mkstemp(suffix=ext, dir=TMP_DIR)
    try:
        written = 0
        with os.fdopen(fd, "wb") as out:
            while chunk := file.file.read(1 << 20):
                written += len(chunk)
                if written > MAX_UPLOAD_BYTES:
                    raise HTTPException(413, _err("upload_too_large", f"file exceeds {MAX_UPLOAD_BYTES} bytes"))
                out.write(chunk)
        if written == 0:
            raise HTTPException(422, {**_err("rejected", "empty upload (0 bytes)"),
                                      "status": "rejected", "reject_reason": "empty upload (0 bytes)"})
        try:
            result = service.detect_file(tmp_path, mode=mode)
        except ValueError as e:
            raise HTTPException(400, _err("bad_preprocessing_mode", str(e)))
        except ServiceBusy as e:
            raise HTTPException(503, _err("busy", str(e)), headers={"Retry-After": "5"})
        except ServiceNotReady as e:
            raise HTTPException(503, _err("model_not_ready", str(e)))
    finally:
        file.file.close()
        try:
            os.remove(tmp_path)
        except FileNotFoundError:
            pass

    result["request_id"] = request_id
    result["upload_bytes"] = written
    result["timing_ms"]["total"] = round((time.perf_counter() - t0) * 1000, 1)
    log.info("detect %s status=%s mode=%s total_ms=%s", request_id, result["status"],
             result["preprocessing"], result["timing_ms"]["total"])
    return JSONResponse(status_code=200 if result["status"] == "ok" else 422, content=result)

In [ ]:
# 3. (cells above) wrote dhwani_infer.py, vcd_service.py, vcd_api.py into APP_DIR
import sys, importlib
sys.path.insert(0, APP_DIR)
import dhwani_infer as dh; importlib.reload(dh)
print("module OK; modes:", dh.PREPROCESSING_MODES, "| pinned revision:", dh.MODEL_REVISION[:7])

In [ ]:
# 4. Locate the pinned checkpoint: HF cache in this runtime -> Drive -> one download
from huggingface_hub import hf_hub_download
MODEL_PATH = None
try:
    MODEL_PATH = hf_hub_download(dh.MODEL_REPO, dh.MODEL_FILE, revision=dh.MODEL_REVISION, local_files_only=True)
    print("Reusing checkpoint already in this runtime's HF cache")
except Exception:
    if USE_DRIVE_CACHE:
        from google.colab import drive
        drive.mount("/content/drive")
        cand = os.path.join(DRIVE_CACHE_DIR, dh.MODEL_FILE)
        if os.path.exists(cand) and os.path.getsize(cand) == dh.MODEL_BYTES:
            MODEL_PATH = cand
            print("Using Drive copy")
    if MODEL_PATH is None:
        print("Not cached in this runtime: downloading pinned checkpoint once (1.26 GB) ...")
        if USE_DRIVE_CACHE:
            MODEL_PATH = hf_hub_download(dh.MODEL_REPO, dh.MODEL_FILE, revision=dh.MODEL_REVISION, local_dir=DRIVE_CACHE_DIR)
        else:
            MODEL_PATH = hf_hub_download(dh.MODEL_REPO, dh.MODEL_FILE, revision=dh.MODEL_REVISION)
assert os.path.getsize(MODEL_PATH) == dh.MODEL_BYTES
print("MODEL_PATH =", MODEL_PATH, "(sha256 is verified by the server at startup)")

In [ ]:
# 5. API key: Colab secret VCD_API_KEY if present, else a random one for this session
import secrets
try:
    from google.colab import userdata
    API_KEY = userdata.get("VCD_API_KEY")
    print("Using Colab secret VCD_API_KEY")
except Exception:
    API_KEY = secrets.token_urlsafe(32)
    print("No VCD_API_KEY secret: generated a session key. Give it to the phones privately;")
    print("it changes when you re-run this cell. Clear this cell's output before sharing the notebook.")
    print("API_KEY =", API_KEY)
assert len(API_KEY) >= 16, "VCD_API_KEY must be at least 16 characters"

In [ ]:
# 6. Start the server (one uvicorn worker, model loaded once) and wait for readiness
import subprocess, time, requests, json
BASE = f"http://127.0.0.1:{PORT}"
if "server" in globals() and server.poll() is None:
    server.terminate(); server.wait(10)
env = {**os.environ, "VCD_API_KEY": API_KEY, "VCD_MODEL_PATH": MODEL_PATH, "VCD_TMP_DIR": TMP_DIR,
       "VCD_PREPROCESSING": DEFAULT_MODE, "VCD_MAX_UPLOAD_BYTES": str(MAX_UPLOAD_MB * 1024 * 1024),
       "VCD_ALLOW_DOWNLOAD": "0"}
server_log = open("/content/vcd_api.log", "w")
server = subprocess.Popen([sys.executable, "-m", "uvicorn", "vcd_api:app", "--app-dir", APP_DIR,
                           "--host", "127.0.0.1", "--port", str(PORT), "--workers", "1"],
                          env=env, stdout=server_log, stderr=subprocess.STDOUT)
health = None
for _ in range(180):
    if server.poll() is not None:
        break
    try:
        health = requests.get(BASE + "/health", timeout=2).json()
        break
    except Exception:
        time.sleep(1)
print(json.dumps(health, indent=2) if health else "server did not answer")
if not health or not health["model_ready"]:
    print(open("/content/vcd_api.log").read()[-3000:])

## Test clips
Upload the **same genuine file used in notebook 01**: `genuine_hi_aman.m4a` (sha256 starts `8b03de09`), which is in the project root on Aman's Mac.
The synthetic clip is re-fetched from MLAAD by its exact pinned path (needs the `HF_TOKEN` Colab secret; MLAAD is CC-BY-NC-4.0, research use only).

In [ ]:
# 7. Get the two baseline clips into TEST_DIR
import shutil, glob
from google.colab import files, userdata
up = files.upload()  # pick genuine_hi_aman.m4a
for n in up:
    shutil.move(n, os.path.join(TEST_DIR, os.path.basename(n)))
syn_rel = "fake/en/Chatterbox/emerald_city_of_oz_20_f000055.wav"
syn_local = hf_hub_download("mueller91/MLAAD", syn_rel, repo_type="dataset",
                            revision="30c3dec763fa4803f11c2df1557d3b0828395366", token=userdata.get("HF_TOKEN"))
shutil.copyfile(syn_local, os.path.join(TEST_DIR, "synthetic__en__Chatterbox__emerald_city_of_oz_20_f000055.wav"))
test_files = sorted(glob.glob(os.path.join(TEST_DIR, "*")))
for p in test_files:
    print(dh.sha256_file(p)[:12], os.path.basename(p))

In [ ]:
# 8. Verify /health and /detect locally (inside Colab) - the backend is not "done" until this passes
import io, threading, numpy as np, soundfile as sf
H = {"X-API-Key": API_KEY}
# Baseline scores from run 20261024T102407Z, keyed by file sha256 (rounded to 4 dp there)
EXPECTED = {
    "8b03de0923525b54aa73cd0c9b940444f652681f9605cfdaa617762970085bae": {"train_peak": 0.0001, "train_zscore": 0.0001, "api_replica": 0.0001},
    "9448a599a4d5cd420191182d20890592e69124c699f704903c22c3614f3fc0ac": {"train_peak": 0.9998, "train_zscore": 0.9901, "api_replica": 0.9916},
}
checks = []
def check(name, cond, detail=""):
    checks.append((name, bool(cond), detail)); print(("PASS " if cond else "FAIL ") + name, detail)

def post(path_or_bytes, filename=None, headers=H, mode=None):
    data = open(path_or_bytes, "rb").read() if isinstance(path_or_bytes, str) else path_or_bytes
    fn = filename or os.path.basename(path_or_bytes)
    return requests.post(BASE + "/detect", headers=headers, params={"mode": mode} if mode else None,
                         files={"file": (fn, data)}, timeout=180)

h = requests.get(BASE + "/health", timeout=10).json()
check("/health model_ready", h["model_ready"], h.get("load_error") or h["load_info"])
check("/health pinned revision", h["model"]["revision"] == dh.MODEL_REVISION)

syn = [p for p in test_files if os.path.basename(p).startswith("synthetic__")][0]
check("no key -> 401", post(syn, headers={}).status_code == 401)
check("wrong key -> 401", post(syn, headers={"X-API-Key": "x" * 32}).status_code == 401)

for p in test_files:
    sha = dh.sha256_file(p)
    for mode in (None, "api_replica"):
        r = post(p, mode=mode); j = r.json()
        used = j.get("preprocessing")
        exp = EXPECTED.get(sha, {}).get(used)
        print(f"  {os.path.basename(p)} mode={used}: HTTP {r.status_code} prediction={j.get('prediction')} "
              f"score={j.get('synthetic_score')} dur={j.get('duration_s')} timing={j.get('timing_ms')}")
        check(f"{os.path.basename(p)} [{used}] HTTP 200", r.status_code == 200, j.get("reject_reason") or "")
        if exp is not None:
            check(f"{os.path.basename(p)} [{used}] matches notebook-01 score", abs(j["synthetic_score"] - exp) <= 0.002,
                  f"{j['synthetic_score']} vs {exp}")
        else:
            print("  (no baseline for this file's sha256 - score not compared)")

r = post(syn, mode="nonsense"); check("unknown mode -> 400", r.status_code == 400, r.text[:120])
r = post(b"hello", filename="notes.txt"); check("unsupported extension -> 415", r.status_code == 415)
try:
    r = post(os.urandom(MAX_UPLOAD_MB * 1024 * 1024 + 1), filename="big.wav")
    check("oversize -> 413", r.status_code == 413, r.status_code)
except requests.exceptions.ConnectionError:
    check("oversize -> 413", True, "server refused before body finished (connection closed after 413)")
buf = io.BytesIO(); sf.write(buf, np.zeros(16000 * 3, np.float32), 16000, format="WAV")
r = post(buf.getvalue(), filename="silent.wav"); check("silent -> 422 rejected", r.status_code == 422 and "silent" in (r.json().get("reject_reason") or ""), r.json().get("reject_reason"))
y, sr = sf.read(syn); buf = io.BytesIO(); sf.write(buf, y[: sr], sr, format="WAV")
r = post(buf.getvalue(), filename="short.wav"); check("1 s clip -> 422 too short", r.status_code == 422 and "too short" in (r.json().get("reject_reason") or ""), r.json().get("reject_reason"))
r = post(b"RIFF\x00\x00garbage" * 50, filename="corrupt.wav"); check("corrupt -> 422 unreadable", r.status_code == 422, r.json().get("reject_reason"))
r = post(b"", filename="empty.wav"); check("empty -> 422", r.status_code == 422)

res = [None, None]
def worker(i): res[i] = post(syn).json()
ts = [threading.Thread(target=worker, args=(i,)) for i in range(2)]
[t.start() for t in ts]; [t.join() for t in ts]
a, b = sorted((x["inference_slot"] for x in res), key=lambda s: s["acquired_monotonic"])
check("2 concurrent requests serialised (no overlap)", a["released_monotonic"] <= b["acquired_monotonic"],
      f"queue_wait_ms={[x['timing_ms']['queue_wait'] for x in res]}")
check("temp upload dir empty after requests", os.listdir(TMP_DIR) == [], os.listdir(TMP_DIR))

n_ok = sum(c[1] for c in checks)
print(f"\nBACKEND LOCAL CHECKS: {n_ok}/{len(checks)} passed")

## 9. Public HTTPS tunnel for the phones

Uses a Cloudflare **quick tunnel** (`cloudflared` 2026.9.3, no account): it prints a random
`https://<words>.trycloudflare.com` URL that forwards to this Colab server.

- The URL is **public**. `/detect` is protected only by the API key. `/health` is open but has no audio and no secrets.
- The URL changes every time the tunnel restarts, and it dies when the Colab runtime stops. Quick tunnels have no uptime guarantee and are for testing only.
- Only send audio you have consent to analyse. Stop the tunnel (step 11) when you're done.

In [ ]:
# 9. Start the tunnel and print the HTTPS base URL
import re
CF = "/content/cloudflared"
CF_URL = "https://github.com/cloudflare/cloudflared/releases/download/2026.9.3/cloudflared-linux-amd64"
if not os.path.exists(CF):
    subprocess.run(["wget", "-q", "-O", CF, CF_URL], check=True)
    os.chmod(CF, 0o755)
print(subprocess.run([CF, "--version"], capture_output=True, text=True).stdout.strip())
if "tunnel" in globals() and tunnel.poll() is None:
    tunnel.terminate(); tunnel.wait(10)
tunnel_log = open("/content/cloudflared.log", "w")
tunnel = subprocess.Popen([CF, "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
                          stdout=tunnel_log, stderr=subprocess.STDOUT)
PUBLIC_URL = None
for _ in range(60):
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("/content/cloudflared.log").read())
    if m:
        PUBLIC_URL = m.group(0); break
    time.sleep(1)
print("PUBLIC_URL =", PUBLIC_URL)

In [ ]:
# 10. Verify the API through the public HTTPS URL (what the phones will use)
ok = False
for _ in range(30):  # new quick-tunnel hostnames can take a few seconds to resolve
    try:
        hj = requests.get(PUBLIC_URL + "/health", timeout=10).json(); ok = True; break
    except Exception:
        time.sleep(2)
print("HTTPS /health:", ok and hj["status"])
for p in test_files:
    r = requests.post(PUBLIC_URL + "/detect", headers=H, files={"file": (os.path.basename(p), open(p, "rb"))}, timeout=180)
    j = r.json()
    print(f"HTTPS /detect {os.path.basename(p)}: HTTP {r.status_code} prediction={j.get('prediction')} "
          f"score={j.get('synthetic_score')} total_ms={j.get('timing_ms', {}).get('total')}")
print("""
Android / curl usage:
  curl -X POST "$URL/detect" -H "X-API-Key: $KEY" -F "file=@voice_note.m4a"
  optional: "$URL/detect?mode=api_replica"
Base URL:""", PUBLIC_URL)

In [ ]:
# 11. Logs / stop everything
print(open("/content/vcd_api.log").read()[-2000:])
STOP = False   # set True and re-run to stop the tunnel and server
if STOP:
    for name in ("tunnel", "server"):
        proc = globals().get(name)
        if proc is not None and proc.poll() is None:
            proc.terminate(); proc.wait(10); print("stopped", name)